load df:

In [1]:
import pandas as pd

#load data from TSV

df = pd.read_csv('tsv/test3.tsv', delimiter='\t')

display(df)

,Measure,Local Onset,Global Onset,Duration,Pitch,MIDI,Voice
0,1,0.0,0.0,4.0,C#3,49,spine_0
1,2,0.0,4.0,2.0,B#2,48,spine_0
2,2,2.0,6.0,2.0,E3,52,spine_0
3,3,0.0,8.0,4.0,D#3,51,spine_0
4,4,0.0,12.0,4.0,G#3,56,spine_0
...,...,...,...,...,...,...,...
1424,115,0.0,456.0,4.0,C#5,73,spine_0
1425,115,0.0,456.0,4.0,G#4,68,spine_0
1426,115,0.0,456.0,4.0,E#4,65,spine_0
1427,115,0.0,456.0,4.0,G#3,56,spine_0


In [2]:
# Binary conversion and visualization (refactored to py_scripts)
import numpy as np
import pandas as pd
from py_scripts.music_utils import create_binary_matrix, plot_binary_matrix

# Configuration
BINARY_RESOLUTION_METHOD = 'manual'   # 'auto' | 'manual' | 'standard'
BINARY_MANUAL_RES = 0.5            # e.g., 0.5 when using 'manual'
Y_AXIS_MODE = 'full'              # 'full' | 'minmax' | 'chroma'
Y_AXIS_MIN = 60                   # optional int when Y_AXIS_MODE=='minmax'
Y_AXIS_MAX = 71                   # optional int when Y_AXIS_MODE=='minmax'

PLOTTING_BACKEND = 'bokeh'  # 'plt' | 'bokeh' | 'none'

# Build binary matrix
binary_matrix_df, meta = create_binary_matrix(
    df,
    resolution_method=BINARY_RESOLUTION_METHOD,
    manual_resolution=BINARY_MANUAL_RES,
    y_mode=Y_AXIS_MODE,
    midi_low=Y_AXIS_MIN,
    midi_high=Y_AXIS_MAX,
    row_order="high_to_low"
)

print(f"Binary matrix shape: {binary_matrix_df.shape}")
print(f"Resolution: {meta['resolution']}, columns: {meta['num_cols']}, y_mode: {meta['y_mode']}")

# Plot using the same backend as parsing cell (if defined), else default to 'plt'
backend_to_use = PLOTTING_BACKEND

# If measure offsets were computed earlier, you can pass them here.
# In this notebook, measure offsets are inside `results` items if needed; we default to None
plot_binary_matrix(
    binary_matrix_df,
    meta,
    backend=backend_to_use,
    measure_offsets=None,
    show_measure_lines=True,
    show=True,
)

Binary matrix shape: (128, 920)
Resolution: 0.5, columns: 920, y_mode: full


figure(id='p1001', ...)

In [3]:
print(binary_matrix_df)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
from py_scripts.binary_matrix_designer import binary_matrix_designer

# Start with an empty 16x16 grid
ui = binary_matrix_designer(
    rows=12,
    cols=12,
    # prototype=None,        # default
    # prototype_meta=None,   # default
    flip_vertical=True,
    display_ui=True         # or False + display(ui)
)


In [12]:
print(binary_matrix_ui)

[[0 0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 1 1]
 [1 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 0]]


In [15]:
# Pattern search metrics: convolution and cross-correlation between binary_matrix_df and binary_matrix_ui
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import Viridis256

STRIDE_Y = 1  # step along pitch axis (rows); increase to prefer octave/tonality jumps
STRIDE_X = 1  # step along time axis (columns)
TOP_N_MATCHES = 20  # number of highest-scoring positions to report per metric
PLOT_THRESHOLD = None  # set to a float (e.g., 0.6) to hide weaker matches in visualisations
MPL_CMAP = 'viridis'

backend_to_use = globals().get('PLOTTING_BACKEND', 'plt')
try:
    output_notebook(hide_banner=True)
except TypeError:
    output_notebook()

if 'binary_matrix_df' not in globals() or 'binary_matrix_ui' not in globals():
    raise NameError('Run the previous cells to define `binary_matrix_df` and `binary_matrix_ui`.')

matrix_source = binary_matrix_df
if isinstance(matrix_source, pd.DataFrame):
    matrix = matrix_source.to_numpy(dtype=float)
    row_labels_full = list(matrix_source.index)
    col_labels_full = list(matrix_source.columns)
else:
    matrix = np.asarray(matrix_source, dtype=float)
    if matrix.ndim != 2:
        raise ValueError('`binary_matrix_df` must be a 2D table or array.')
    row_labels_full = list(range(matrix.shape[0]))
    col_labels_full = list(range(matrix.shape[1]))

kernel_candidate = binary_matrix_ui
if isinstance(kernel_candidate, pd.DataFrame):
    kernel_array = kernel_candidate.to_numpy(dtype=float)
else:
    kernel_array = np.asarray(kernel_candidate, dtype=float)

if kernel_array.ndim != 2:
    raise ValueError('`binary_matrix_ui` must be a 2D structure to act as a kernel.')

if kernel_array.size == 0:
    raise ValueError('`binary_matrix_ui` is empty; draw a pattern before running the analysis cell.')

if kernel_array.shape[0] > matrix.shape[0] or kernel_array.shape[1] > matrix.shape[1]:
    raise ValueError('Kernel larger than source matrix; adjust the UI export or reduce resolution.')

window_shape = kernel_array.shape
out_rows = matrix.shape[0] - window_shape[0] + 1
out_cols = matrix.shape[1] - window_shape[1] + 1
if out_rows <= 0 or out_cols <= 0:
    raise ValueError('Kernel cannot slide within the source matrix; check dimensions.')

windows = np.lib.stride_tricks.sliding_window_view(matrix, window_shape)
conv_scores_full = (windows * kernel_array).sum(axis=(-2, -1))

kernel_weight = kernel_array.sum()
if kernel_weight != 0:
    conv_norm_full = conv_scores_full / kernel_weight
else:
    conv_norm_full = conv_scores_full.astype(float)

windows_flat = windows.reshape(out_rows, out_cols, -1)
kernel_flat = kernel_array.reshape(-1)
kernel_mean = kernel_flat.mean()
kernel_zero_mean = kernel_flat - kernel_mean
kernel_norm = np.linalg.norm(kernel_zero_mean)

window_means = windows_flat.mean(axis=2, keepdims=True)
windows_zero_mean = windows_flat - window_means
window_norms = np.linalg.norm(windows_zero_mean, axis=2)
cross_cov_full = np.tensordot(windows_zero_mean, kernel_zero_mean, axes=([2], [0]))
denominator = window_norms * kernel_norm
norm_cross_full = np.divide(cross_cov_full, denominator, out=np.zeros_like(cross_cov_full), where=denominator > 0)

row_positions = list(range(0, out_rows, max(1, int(STRIDE_Y))))
col_positions = list(range(0, out_cols, max(1, int(STRIDE_X))))

conv_scores = conv_scores_full[np.ix_(row_positions, col_positions)]
conv_norm = conv_norm_full[np.ix_(row_positions, col_positions)]
cross_cov = cross_cov_full[np.ix_(row_positions, col_positions)]
norm_cross = norm_cross_full[np.ix_(row_positions, col_positions)]

row_labels = [row_labels_full[pos] for pos in row_positions]
col_labels = [col_labels_full[pos] for pos in col_positions]

conv_norm_df = pd.DataFrame(conv_norm, index=row_labels, columns=col_labels)
conv_raw_df = pd.DataFrame(conv_scores, index=row_labels, columns=col_labels)
cross_cov_df = pd.DataFrame(cross_cov, index=row_labels, columns=col_labels)
norm_cross_df = pd.DataFrame(norm_cross, index=row_labels, columns=col_labels)

def summarize_best(name, data, labels_y, labels_x):
    finite_mask = np.isfinite(data)
    if not finite_mask.any():
        print(f'{name}: no finite values to summarise.')
        return
    flat_index = np.nanargmax(data)
    r, c = divmod(flat_index, data.shape[1])
    print(f'{name} best: {data[r, c]:.3f} at top-left (row={labels_y[r]}, col={labels_x[c]})')

summarize_best('Normalised overlap', conv_norm, row_labels, col_labels)
summarize_best('Cross-covariance', cross_cov, row_labels, col_labels)
summarize_best('Normalised cross-correlation', norm_cross, row_labels, col_labels)

def top_matches(name, data, labels_y, labels_x, top_n, threshold=None):
    if top_n is None or top_n <= 0:
        return
    flat = data.reshape(-1)
    mask = np.isfinite(flat)
    if threshold is not None:
        mask &= flat >= threshold
    indices = np.nonzero(mask)[0]
    if indices.size == 0:
        print(f'{name}: no entries meet the threshold.')
        return
    ranked = indices[np.argsort(flat[indices])[::-1]]
    limit = min(top_n, ranked.size)
    print(f'{name} top {limit} positions:')
    for idx in ranked[:limit]:
        val = flat[idx]
        r, c = divmod(idx, data.shape[1])
        print(f'  score={val:.3f} at row={labels_y[r]}, col={labels_x[c]}')

top_matches('Normalised overlap', conv_norm, row_labels, col_labels, TOP_N_MATCHES, PLOT_THRESHOLD)
top_matches('Cross-covariance', cross_cov, row_labels, col_labels, TOP_N_MATCHES, PLOT_THRESHOLD)
top_matches('Normalised cross-correlation', norm_cross, row_labels, col_labels, TOP_N_MATCHES, PLOT_THRESHOLD)

def apply_threshold(data, threshold):
    if threshold is None:
        return data
    return np.where(data >= threshold, data, np.nan)

plot_series = [
    ('Normalised overlap', conv_norm),
    ('Cross-covariance', cross_cov),
    ('Normalised cross-correlation', norm_cross),
]

def plot_matplotlib_single(title, data):
    fig, ax = plt.subplots(figsize=(9, 6))
    masked = apply_threshold(data, PLOT_THRESHOLD)
    im = ax.imshow(masked, cmap=MPL_CMAP, origin='upper', aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('time offset (columns)')
    ax.set_ylabel('pitch offset (rows)')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

def finite_bounds(data):
    finite = data[np.isfinite(data)]
    if finite.size == 0:
        return 0.0, 1.0
    low = float(finite.min())
    high = float(finite.max())
    if low == high:
        high = low + 1e-9
    return low, high

def plot_bokeh_single(title, data):
    masked = apply_threshold(data, PLOT_THRESHOLD)
    low, high = finite_bounds(masked)
    fig = figure(title=title, x_range=(0, masked.shape[1]), y_range=(0, masked.shape[0]), width=900, height=600, tools='pan,wheel_zoom,reset,save')
    mapper = LinearColorMapper(palette=Viridis256, low=low, high=high)
    fig.image(image=[np.flipud(masked)], x=0, y=0, dw=masked.shape[1], dh=masked.shape[0], color_mapper=mapper)
    fig.add_layout(ColorBar(color_mapper=mapper), 'right')
    fig.xaxis.axis_label = 'time offset (columns)'
    fig.yaxis.axis_label = 'pitch offset (rows, top=high)'
    show(fig)

if backend_to_use == 'plt':
    for title, data in plot_series:
        plot_matplotlib_single(title, data)
elif backend_to_use == 'bokeh':
    for title, data in plot_series:
        plot_bokeh_single(title, data)
elif backend_to_use == 'both':
    for title, data in plot_series:
        plot_matplotlib_single(title, data)
        plot_bokeh_single(title, data)
elif backend_to_use == 'none':
    pass
else:
    print(f'Unknown backend {backend_to_use!r}. Available options: "plt", "bokeh", "both", "none". Defaulting to Matplotlib.')
    for title, data in plot_series:
        plot_matplotlib_single(title, data)

Normalised overlap best: 0.800 at top-left (row=50, col=532)
Cross-covariance best: 5.500 at top-left (row=55, col=840)
Normalised cross-correlation best: 0.798 at top-left (row=52, col=376)
Normalised overlap top 20 positions:
  score=0.800 at row=52, col=348
  score=0.800 at row=59, col=904
  score=0.800 at row=61, col=808
  score=0.800 at row=61, col=252
  score=0.800 at row=64, col=288
  score=0.800 at row=57, col=564
  score=0.800 at row=56, col=560
  score=0.800 at row=76, col=772
  score=0.800 at row=55, col=840
  score=0.800 at row=50, col=532
  score=0.800 at row=61, col=780
  score=0.800 at row=57, col=820
  score=0.800 at row=64, col=796
  score=0.800 at row=78, col=228
  score=0.800 at row=57, col=836
  score=0.800 at row=66, col=272
  score=0.700 at row=61, col=807
  score=0.700 at row=50, col=533
  score=0.700 at row=64, col=795
  score=0.700 at row=56, col=684
Cross-covariance top 20 positions:
  score=5.500 at row=66, col=272
  score=5.500 at row=55, col=840
  score=5.2

### Pattern Search Metrics Overview

- **Normalised overlap** divides the raw overlap count by the sum of kernel values. With binary kernels this ranges from 0 (no overlap) to 1 (perfect alignment). Values above 1 only appear if the kernel contains weights greater than 1.
- **Cross-covariance** subtracts the local mean before scoring. The scale depends on the data magnitude; higher positive numbers indicate stronger alignment with the kernel's on/off pattern, while negative values suggest an inverted match.
- **Normalised cross-correlation** rescales cross-covariance by both vector norms, yielding scores in [-1, 1]. A value of 1 is a perfect positive match, 0 indicates no linear relationship, and -1 is a perfect inversion.
- `STRIDE_Y` / `STRIDE_X` control how densely the kernel slides. Increase `STRIDE_Y` to favour octave-spaced matches, or `STRIDE_X` to skip columns for faster scans.
- `TOP_N_MATCHES` prints the strongest matches per metric after optional thresholding, giving you a ranked list of candidate alignments.
- `PLOT_THRESHOLD` masks values below the chosen cutoff in both Matplotlib and Bokeh views. Set it to `None` to show everything, or to a number (e.g., 0.6) to highlight only strong responses.